In [5]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import concurrent.futures
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
from tqdm import tqdm  # Progress bar
import time

In [6]:
dummy_url = 'https://www.zameen.com/Property/f_11_f_11_markaz_f_11_size_3700_square_feet_karakoram_enclave_4_bedrooms_apartment_for_sale-6282364-3286-1.html'

In [7]:
t1 = time.time()
response = requests.get(dummy_url)
if response.status_code == 200:
    html_content = response.content
    soup = BeautifulSoup(html_content,'lxml')
    tag = soup.find('span', class_="_2fdf7fc5", attrs={'aria-label': 'Area'})
    print(tag.get_text())

t2 = time.time()

delt = t2-t1

round(delt,2)

16.4 Marla


1.26

In [8]:
df = pd.read_csv('output.csv')
df.head()

,property_type,purpose,baths,bedrooms,location,sqft,price,page_url
0,House,For Sale,7,6,Cantt,21780.000,4500.0,https://www.zameen.com/Property/faisal_cantonm...
1,House,For Sale,8,6,Gulistan-e-Jauhar,4356.016,350.0,https://www.zameen.com/Property/karachi_gulist...
2,Flat,For Sale,3,3,DHA Defence,24230.339,210.0,https://www.zameen.com/Property/d_h_a_dha_phas...
3,House,For Sale,1,2,Malir,8712.032,65.0,https://www.zameen.com/Property/malir_malir_ka...
4,House,For Sale,4,4,Gadap Town,26136.096,130.0,https://www.zameen.com/Property/surjani_town_s...


In [18]:
urls = df['page_url']
urls.shape

(59247,)

In [19]:
#OLD


# # Function to fetch and parse HTML content
# def fetch_area_value(url):
#     try:
#         response = requests.get(url)
#         if response.status_code == 200:
#             html_content = response.content
#             soup = BeautifulSoup(html_content, 'lxml')
#             tag = soup.find('span', class_="_2fdf7fc5", attrs={'aria-label': 'Area'})
#             return tag.get_text() if tag else None
#     except Exception:
#         return None  # Return None on error
    
# # Function to fetch area values in parallel
# def fetch_area_values_parallel(urls):
#     results = [None] * len(urls)  # Initialize results list
#     with concurrent.futures.ThreadPoolExecutor() as executor:
#         futures = {executor.submit(fetch_area_value, url): index for index, url in enumerate(urls)}
        
#         for future in concurrent.futures.as_completed(futures):
#             index = futures[future]
#             try:
#                 results[index] = future.result()
#             except Exception:
#                 results[index] = None  # Handle unexpected errors

#     return results

In [20]:
# Create a session with retries and timeouts
session = requests.Session()
retry = Retry(total=3, backoff_factor=0.3, status_forcelist=[500, 502, 503, 504])
adapter = HTTPAdapter(max_retries=retry)
session.mount("http://", adapter)
session.mount("https://", adapter)

# Function to fetch and parse HTML content
def fetch_area_value(url):
    try:
        response = session.get(url, timeout=20)  # Set a timeout to avoid hanging
        if response.status_code == 200:
            html_content = response.content
            soup = BeautifulSoup(html_content, 'lxml')
            tag = soup.find('span', class_="_2fdf7fc5", attrs={'aria-label': 'Area'})
            return tag.get_text() if tag else None
        else:
            return None
    except Exception:
        return None  # Return None on error

# Function to fetch area values in parallel
def fetch_area_values_parallel(urls, max_workers=20):  # Increase the number of workers
    results = [None] * len(urls)  # Initialize results list
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_area_value, url): index for index, url in enumerate(urls)}
        
        # Use tqdm to show progress bar
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(urls), desc="Fetching area values"):
            index = futures[future]
            try:
                results[index] = future.result()
            except Exception:
                results[index] = None  # Handle unexpected errors

    return results

In [21]:
t1 = time.time()
# areas = fetch_area_values_parallel(urls) ## webscraping
t2 = time.time()

round(t2-t1,2)

Fetching area values: 100%|████████████████████████████████████████████████████| 59247/59247 [1:26:41<00:00, 11.39it/s]


5204.74

took ~82mint

In [24]:
areas

['2,000 Sq. Yd.',
 '400 Sq. Yd.',
 '222 Sq. Yd.',
 '106 Sq. Yd.',
 '240 Sq. Yd.',
 '100 Sq. Yd.',
 None,
 '500 Sq. Yd.',
 '240 Sq. Yd.',
 '256 Sq. Yd.',
 '286 Sq. Yd.',
 '286 Sq. Yd.',
 '286 Sq. Yd.',
 '328 Sq. Yd.',
 '222 Sq. Yd.',
 '120 Sq. Yd.',
 '159 Sq. Yd.',
 '134 Sq. Yd.',
 '81 Sq. Yd.',
 '335 Sq. Yd.',
 '286 Sq. Yd.',
 '286 Sq. Yd.',
 '330 Sq. Yd.',
 '350 Sq. Yd.',
 '500 Sq. Yd.',
 '350 Sq. Yd.',
 '350 Sq. Yd.',
 '500 Sq. Yd.',
 '350 Sq. Yd.',
 '500 Sq. Yd.',
 None,
 '350 Sq. Yd.',
 None,
 '500 Sq. Yd.',
 '500 Sq. Yd.',
 None,
 None,
 '240 Sq. Yd.',
 '150 Sq. Yd.',
 '600 Sq. Yd.',
 '100 Sq. Yd.',
 '153 Sq. Yd.',
 '120 Sq. Yd.',
 '120 Sq. Yd.',
 '600 Sq. Yd.',
 '500 Sq. Yd.',
 '242 Sq. Yd.',
 '600 Sq. Yd.',
 '243 Sq. Yd.',
 '228 Sq. Yd.',
 '150 Sq. Yd.',
 '600 Sq. Yd.',
 '1,000 Sq. Yd.',
 '350 Sq. Yd.',
 '120 Sq. Yd.',
 '288 Sq. Yd.',
 '350 Sq. Yd.',
 '500 Sq. Yd.',
 '500 Sq. Yd.',
 '500 Sq. Yd.',
 '194 Sq. Yd.',
 '400 Sq. Yd.',
 '138 Sq. Yd.',
 '240 Sq. Yd.',
 '350 Sq. Yd.',
 '

In [26]:
import numpy as np

In [27]:
np_areas = np.array(areas)

In [31]:
df['area'] = np_areas

In [47]:
def separate_value_unit(s):
    if s is None:
        return None,None
    parts = s.split()  # Use regular split for strings
    value = parts[0]
    unit = ' '.join(parts[1:])  # Join the remaining parts for the unit
    return value, unit

In [48]:
df['val'], df['unit'] = zip(*df['area'].apply(separate_value_unit))

In [51]:
df[['sqft','area','val','unit']]

,sqft,area,val,unit
0,21780.000,"2,000 Sq. Yd.","2,000",Sq. Yd.
1,4356.016,400 Sq. Yd.,400,Sq. Yd.
2,24230.339,222 Sq. Yd.,222,Sq. Yd.
3,8712.032,106 Sq. Yd.,106,Sq. Yd.
4,26136.096,240 Sq. Yd.,240,Sq. Yd.
...,...,...,...,...
59242,26136.096,240 Sq. Yd.,240,Sq. Yd.
59243,2178.008,200 Sq. Yd.,200,Sq. Yd.
59244,26136.096,240 Sq. Yd.,240,Sq. Yd.
59245,21235.578,194 Sq. Yd.,194,Sq. Yd.


In [62]:
bkp_df = df.copy(deep=True)

In [63]:
bkp_df.to_csv('backup.csv', index=False, sep=',', encoding='utf-8')

In [65]:
df.dropna(inplace = True)

In [68]:
df['unit'].unique()

array(['Sq. Yd.', 'Marla'], dtype=object)

In [71]:
def convert_to_sqft(val, unit):
    val = float(val.replace(',',''))  # Convert value to float for calculations
    
    if unit.lower() == "sq. yd." or unit.lower() == "sqyd":
        # 1 square yard = 9 square feet
        return val * 9
    elif unit.lower() == "marla":
        # 1 marla = 272.25 square feet
        return val * 272.25
    else:
        raise ValueError(f"Unknown unit: {unit}")

In [72]:
df['area_sqft'] =  df.apply(lambda row: convert_to_sqft(row['val'], row['unit']), axis=1)

In [86]:
df[['sqft','area','area_sqft']]

,sqft,area,area_sqft
0,21780.000,"2,000 Sq. Yd.",18000.0
1,4356.016,400 Sq. Yd.,3600.0
2,24230.339,222 Sq. Yd.,1998.0
3,8712.032,106 Sq. Yd.,954.0
4,26136.096,240 Sq. Yd.,2160.0
...,...,...,...
59242,26136.096,240 Sq. Yd.,2160.0
59243,2178.008,200 Sq. Yd.,1800.0
59244,26136.096,240 Sq. Yd.,2160.0
59245,21235.578,194 Sq. Yd.,1746.0


In [87]:
df.to_csv('corrected_areas.csv', index=False, sep=',', encoding='utf-8')